## Helper functions and packages

In [1]:
# functions

from ensemble_chaos_tools import EnsembleChaos
from ensemble_chaos_tools import fix_lat_lon
from ensemble_chaos_tools import check_chaos
from scipy.optimize import curve_fit
from earthkit.regrid import interpolate
import xarray as xr
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np
import nicopal as ncp
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask.diagnostics import ProgressBar
import pickle
import pandas as pd

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="jpeg"

In [2]:
# patterns = [
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/*/t2m.*07.as1e5.GLOBAL_025.nc",
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/*/t2m.*08.as1e5.GLOBAL_025.nc",
#     "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/*/t2m.*09.as1e5.GLOBAL_025.nc",
# ]

# paths_t2m_for_climato = []
# for pattern in patterns:
#     files = glob.glob(pattern)
#     for file in files:
#         year_val = int(file.split("/")[7])
#         if year_val > 1990:
#             paths_t2m_for_climato.append(file)

# era5_t2m_for_climato = xr.open_mfdataset(
#     paths_t2m_for_climato, parallel=True, engine="h5netcdf"
# )

# era5_t2m_for_climato = era5_t2m_for_climato.sel(
#     time=era5_t2m_for_climato.time.dt.hour.isin([0, 6, 12, 18])
# )


# era5_2t_climato = era5_t2m_for_climato.groupby(["time.month", "time.hour"]).mean("time")

# with ProgressBar():
#     era5_2t_climato.to_netcdf(
#         "/homedata/pchevali/ERA5_CLIMATO/t2m_era5_climato.nc", engine="h5netcdf"
#     )

In [3]:
with open("/homedata/pchevali/n320_coordinates.pkl","rb") as f:
    grid=pickle.load(f)

In [4]:
climato_era5=xr.open_dataset("/homedata/pchevali/ERA5_CLIMATO/t2m_era5_climato.nc")
climato_era5=climato_era5.isel(hour=climato_era5.hour.isin([12])).mean("month")
climato_era5=interpolate(climato_era5["t2m"].values,{"grid": [0.25,0.25]}, {"grid": "N320"})
climato_era5 = xr.DataArray(
    data=climato_era5,
    dims=["values"],
    coords={
        "latitude": ("values", grid["latitude"]),
        "longitude": ("values", grid["longitude"])
    },
)
climato_era5=climato_era5.assign_coords(longitude=(((climato_era5.longitude + 180) % 360) - 180)).values
def plot_composite(
    field1,
    cmap_legend,
    lat,
    lon,
    steps,
    members,
    region,
    field2=None,
    title=None,
    save=None,
):
    fig, ax = plt.subplots(
        figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree()}
    )

    cf = ax.tricontourf(
        lon,
        lat,
        field1.sel(step=field1.step.isin(steps), number=field1.number.isin(members))
        .mean(["step", "number"])
        .values-climato_era5[region],
        levels=30,
        cmap=ncp.pal("Vanadium"),
        transform=ccrs.PlateCarree(),
    )

    cbar = fig.colorbar(cf, ax=ax, shrink=0.9)
    cbar.ax.tick_params(labelsize="large")
    cbar.set_label(cmap_legend, size="x-large")

    if field2 is not None:
        contours = ax.tricontour(
            lon,
            lat,
            (
                field2.sel(
                    step=field2.step.isin(steps), number=field2.number.isin(members)
                )
                / 9.81
            )
            .mean(["step", "number"])
            .values,
            transform=ccrs.PlateCarree(),
            levels=15,
            colors="black",
        )

        ax.clabel(
            contours,
            fmt="%.0f",
            fontsize="large",
            colors="black",
            inline=True,
            use_clabeltext=True,
        )

    ax.coastlines(alpha=0.7)

    plt.title(title)

    if save is not None:
        plt.savefig(f"composite_maps/composite-{save}.pdf", bbox_inches="tight")

    plt.show()


def plot_extreme_events_composite(
    datasets_results,
    target,
    region,
    n=5,
    climato=0,
    date_key="2003072300",
    var_main="2t",
    var_contour="z_500",
    save=None,
):
    """
    wrapper
    """

    # Extract and convert temperature for the specific point
    point_data = datasets_results[date_key][var_main].isel(values=target)

    # find the top n steps and members
    stacked_point_data = point_data.stack(sample=["step", "number"])

    sorted_data = stacked_point_data.sortby(stacked_point_data, ascending=False)
    selected_samples = sorted_data.isel(sample=slice(0, n))
    plot_title = f"Top {n} extreme events averaged"

    selected_steps = xr.DataArray(selected_samples.step.values, dims="temp")
    selected_members = xr.DataArray(selected_samples.number.values, dims="temp")

    field1 = datasets_results[date_key][var_main].isel(values=region)

    field2 = datasets_results[date_key][var_contour].isel(values=region)

    plot_composite(
        field1=field1,
        cmap_legend="Temperature anomaly",
        lat=LAT[region],
        lon=LON[region],
        steps=selected_steps,
        members=selected_members,
        field2=field2,
        save=save,
        region=region
    )

/home/pchevali/.local/lib/python3.12/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'argo' loading failed:
cannot import name 'collections_to_dsk' from 'dask.base' (/home/pchevali/.local/lib/python3.12/site-packages/dask/base.py)
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


In [5]:
def get_nearest_point(ds, target_lat, target_lon):
    distances = (ds["latitude"] - target_lat) ** 2 + (ds["longitude"] - target_lon) ** 2
    return distances.argmin().compute().item()

In [6]:
def growth_rate_pairwise_bootstrap(data, space_indexes, times=[12],N=100, save=None, crps=False):
        """Estimate the upper-bound predictability limit and growth rate (alpha) over a spatial region.

        Unlike lyapunov_over_area_pairwise which uses a linear fit on log-RMSE,
        this function computes the root-mean-square error (RMSE) for all unique pairs
        and fits the Lorenz logistic growth model (dE/dt = alpha * E * (1 - E/E_inf))
        to estimate the error growth rate (alpha) and the saturation error (E_inf).

        Args:
            lat: Slice or array of latitudes defining the region to average over.
            lon: Slice or array of longitudes defining the region to average over.
            times: List of hours to filter the data by. Defaults to [0,6,12,18].

        Returns:
            alpha, E_inf: Estimated growth rate (days^-1) and saturation error, rounded to 3
                          decimal places. Also displays a matplotlib figure and prints the result.
        """
        time_mask = data.valid_time.dt.hour.isin(times)
    
        #the first input is to same if using crps
        if crps:
            time_mask[0]=False

        # select only the area on the point we're interested in and times we want
        data = data.isel(values=space_indexes).sel(
            step=time_mask
        ).load()
        
        # computation of logdist
        data = data.transpose("step", "number", "values")

        # Subtract the ensemble mean to center the data around 0 (trying to avoid catastrophic annulation)
        data = data - data.mean(dim="number", skipna=True)
        
        # weights
        weights = np.cos(np.deg2rad(data.latitude.values))
        weights_norm = weights / np.mean(weights)

        # compute RMSE pairwise relatively fast
        v = data.values #extract numpy array
        sq_mean = np.mean((v**2) * weights_norm, axis=-1) #compute the squared mean of the data for an ensemble and step [step x number]
        # dark magic for the compute of the cross term when expanding the square, first apply the weights on one of the terms then
        # perform a batched matrix multiplication (@) to calculate the dot product of every ensemble pair (2AB) acrros all time steps
        #the final .transpose(1, 2, 0) transposes the result from (step, number, number) to (number, number, step) so that tri_indices works fine
        rmse = np.sqrt(np.maximum((sq_mean[:, :, None] + sq_mean[:, None, :] - 2 * ((v * weights_norm) @ v.transpose(0, 2, 1)) / v.shape[-1]).transpose(1, 2, 0), 0))
        #max in case matmul gives smth <0, happens sometimes when numbers are very small

        # Pairwise extractions and then mean over all ensemble pairs
        pairwise_rmse = rmse[np.triu_indices(rmse.shape[0], k=1)]
        logdist = np.log(pairwise_rmse)
        mean_rmse = np.mean(pairwise_rmse, axis=0)
        mean_logdist = np.mean(logdist, axis=0)

        # extract times for better plotting
        time_indexes = data.valid_time.values
        time_since_start = (time_indexes - time_indexes[0]).astype(
            "timedelta64[h]"
        ).astype(float) / 24.0

        if crps:
            time_since_start=time_since_start+0.25
        
        # fit lorenz model
        # solution of lorenz model for growth
        E0 = mean_rmse[0]

        def logistic_solution(t, alpha_param, E_inf_param):
            return E_inf_param / (
                1.0 + ((E_inf_param - E0) / E0) * np.exp(-alpha_param * t)
            )

        max_E = np.max(mean_rmse)

        popt, _ = curve_fit(
            logistic_solution,
            time_since_start,
            mean_rmse,
            p0=[0.3, max_E],
            bounds=([0.0, 0.0], [1, max_E * 5.0]),
        )

        alpha = np.round(popt[0], 3)
        E_inf = np.round(popt[1], 3)

        #bootstrap (the first fit was done to warmup the bootstraps)

        num_pairs = pairwise_rmse.shape[0]
        bootstrap_alphas, bootstrap_E_infs = [], []

        for n in range(N):
            idx = np.random.choice(num_pairs, num_pairs, replace=True)
            bootstrap_mean_rmse = np.mean(pairwise_rmse[idx], axis=0)

            popt_bootstrap, _ = curve_fit(
                    logistic_solution,
                    time_since_start,
                    bootstrap_mean_rmse,
                    p0=[alpha, E_inf], 
                    bounds=([0.0, 0.0], [1, max_E * 5.0]),
                    maxfev=2000 
                )
            bootstrap_alphas.append(popt_bootstrap[0])
            bootstrap_E_infs.append(popt_bootstrap[1])

        alpha_ci_plusminus = 1.96 * np.std(bootstrap_alphas)
        alpha_lower = np.round(alpha - alpha_ci_plusminus, 3)
        alpha_upper = np.round(alpha + alpha_ci_plusminus, 3)

        E_ci_plusminus = 1.96 * np.std(bootstrap_E_infs)
        E_inf_lower = np.round(E_inf - E_ci_plusminus, 3)
        E_inf_upper = np.round(E_inf + E_ci_plusminus, 3)
        
        # plot and print
        plt.style.use("seaborn-v0_8-whitegrid")
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.set_ylabel("$log(RMSE)$",fontsize="xx-large")
        ax.set_xlabel("Day",fontsize="xx-large")

        for n in range(logdist.shape[0]):
            label = "Pairwise Differences" if n == 0 else None
            ax.plot(
                time_since_start,
                logdist[n, :],
                color="grey",
                alpha=0.5,
                linewidth=1,
                zorder=1,
                label=label,
            )

        ax.plot(
            time_since_start,
            mean_logdist,
            "o-",
            color="black",
            linewidth=2,
            zorder=2,
            label="Mean $log(RMSE)$",
        )

        # fit for plotting
        E_theoretical = logistic_solution(time_since_start, alpha, E_inf)
        log_E_theoretical = np.log(E_theoretical)

        ax.plot(
            time_since_start,
            log_E_theoretical,
            color="blue",
            linewidth=2,
            ls="--",
            zorder=3,
            label=f"Fit (α $\in$ [{alpha_lower},{alpha_upper}])",
        )

        #ax.set_title("Log Differences (Pairwise)",fontsize="xx-large")
        ax.legend(fontsize="xx-large",loc="lower right")
        ax.tick_params(axis='both', which='major', labelsize="x-large")
        plt.tight_layout()
        if save is not None:
            plt.savefig(f"lyap_plots/{save}.png",dpi=300)
        plt.show()

        print(f"Estimated Growth Rate (α): {alpha} [95% CI: {alpha_lower}, {alpha_upper}] days^-1 | "
              f"Saturation Error (E_inf): {E_inf} [95% CI: {E_inf_lower}, {E_inf_upper}]")

        #return alpha, E_inf, (alpha_lower, alpha_upper), (E_inf_lower, E_inf_upper)

In [7]:
def moist_static_energy(T_s,T_500,T_2d,sp,z_500):
    #constants
    c_p = 1004.0 # Specific heat of air at constant pressure (J/kg/K)
    L_v = 2.5e6 # Latent heat of vaporization (J/kg)
    epsilon = 0.622 # Molar ratio of water vapor to dry air
    
    def calc_vapor_pressure(T_kelvin):
        #tetens formula (gives kpa but we want pa)
        T_celsius = T_kelvin - 273.15
        e_hpa = 6.1094 * np.exp((17.625 * T_celsius) / (T_celsius + 243.04))
        return e_hpa*100

    e_actual = calc_vapor_pressure(T_2d)
    q_s = epsilon * (e_actual / sp)
    MSE_s = (c_p * T_s) + (L_v * q_s) + z["z"]


    e_sat_500 = calc_vapor_pressure(T_500)
    q_sat_500 = epsilon * (e_sat_500 / 50000)
    MSE_500_star = (c_p * T_500) + (L_v * q_sat_500) + z_500

    return MSE_500_star, MSE_s

def get_highest_index(data,n=5):
    max_data = data.max(dim="step")
    top = max_data.sortby(max_data, ascending=False).head(number=n)
    return top.number.values

Figure out the mask for outputs of aifs

In [8]:
mask = (grid['latitude']>15)*(grid['latitude']<75)*((grid['longitude']>300)+(grid['longitude']<30))
with open("/homedata/pchevali/mask_eu.pkl","wb") as f:
    pickle.dump(mask,f)

## Import ERA5

In [9]:
era5_202506_07 = xr.open_mfdataset(
    [
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2025/t2m.202506.as1e5.GLOBAL_025.nc",
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2025/t2m.202507.as1e5.GLOBAL_025.nc",
    ],
    engine="h5netcdf"
)
era5_202506_07 = era5_202506_07.sel(
    time=era5_202506_07.time.dt.hour.isin([0, 6, 12, 18])
).compute()
mask = (
    era5_202506_07.time >= np.datetime64("2025-06-20T00:00:00.000000000")
).values * (
    era5_202506_07.time <= np.datetime64("2025-07-20T00:00:00.000000000")
).values
era5_202506_07 = fix_lat_lon(era5_202506_07.isel(time=mask))

In [10]:
era5_200307_08 = xr.open_mfdataset(
    [
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2003/t2m.200307.as1e5.GLOBAL_025.nc",
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2003/t2m.200308.as1e5.GLOBAL_025.nc",
    ],
    engine="h5netcdf"
)
era5_200307_08 = era5_200307_08.sel(
    time=era5_200307_08.time.dt.hour.isin([0, 6, 12, 18])
).compute()
mask = (
    era5_200307_08.time >= np.datetime64("2003-07-23T00:00:00.000000000")
).values * (
    era5_200307_08.time <= np.datetime64("2003-08-22T00:00:00.000000000")
).values
era5_200307_08 = fix_lat_lon(era5_200307_08.isel(time=mask))

In [11]:
era5_201907_08 = xr.open_mfdataset(
    [
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2019/t2m.201907.as1e5.GLOBAL_025.nc",
        "/bdd/ERA5/NETCDF/GLOBAL_025/hourly/AN_SF/2019/t2m.201908.as1e5.GLOBAL_025.nc",
    ],
    engine="h5netcdf"
)
era5_201907_08 = era5_201907_08.sel(
    time=era5_201907_08.time.dt.hour.isin([0, 6, 12, 18])
).compute()
mask = (
    era5_201907_08.time >= np.datetime64("2019-07-13T00:00:00.000000000")
).values * (
    era5_201907_08.time <= np.datetime64("2019-08-12T00:00:00.000000000")
).values
era5_201907_08 = fix_lat_lon(era5_201907_08.isel(time=mask))

## Import the AIFS-CRPS runs

In [12]:
z = xr.open_dataset(
    "/scratchx/pchevali/AIFS_OUTPUTS_REGRIDDED/2019071300-boosting_2019_paris_record/aifs_ensemble-2019071300-boosting_2019_paris_record-z.nc",
)

In [13]:
dates = ["2003072300", "2019071300", "2025062000"]
variables = ["2d", "2t", "sp", "t_500", "tp", "z_500"]

base_dir = "/scratchx/pchevali/AIFS_OUTPUTS_REGRIDDED"

datasets_results = {}

for date in dates:
    datasets_results[date] = {var: {} for var in variables}

    for var in variables:
        pattern = f"{base_dir}/{date}-*/*-{var}.nc"
        file_paths = glob.glob(pattern)

        ds = xr.open_dataset(file_paths[0], decode_timedelta=True,chunks="auto",engine="h5netcdf")
        ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
        datasets_results[date][var] = ds[var]

for date in dates:
    datasets_results[date]=xr.merge(datasets_results[date].values(),compat="override")

In [14]:
LON=datasets_results['2003072300']['z_500'].longitude.values 
LAT=datasets_results['2003072300']['z_500'].latitude.values 

## Usual points and regions

In [15]:
france_latitude_bnd = slice(42, 51)
france_longitude_bnd = slice(-5, 8)

paris_latitude = 49
paris_longitude = 2.5

bc_latitude_bnd = slice(48, 60)
bc_longitude_bnd = slice(-139, -114)
lytton_latitude = 50.23
lytton_longitude = -121.58

northern_hemisphere_latitude_bnd = slice(0, 90)
northern_hemisphere_longitude_bnd = slice(-180, 180)

france_centered_latitude_bnd = slice(15, 70)
france_centered_longitude_bnd = slice(-60, 60)

plot_region_latitude_bnd = slice(30, 60)
plot_region_longitude_bnd = slice(-20, 30)

france_indexs = (
    (LAT >= france_latitude_bnd.start)
    & (LAT <= france_latitude_bnd.stop)
    & (LON >= france_longitude_bnd.start)
    & (LON <= france_longitude_bnd.stop)
)

france_centered_indexs = (
    (LAT >= france_centered_latitude_bnd.start)
    & (LAT <= france_centered_latitude_bnd.stop)
    & (LON >= france_centered_longitude_bnd.start)
    & (LON <= france_centered_longitude_bnd.stop)
)

plot_region_indexs = (
    (LAT >= plot_region_latitude_bnd.start)
    & (LAT <= plot_region_latitude_bnd.stop)
    & (LON >= plot_region_longitude_bnd.start)
    & (LON <= plot_region_longitude_bnd.stop)
)

paris_index = get_nearest_point(
    datasets_results["2003072300"]["z_500"], paris_latitude, paris_longitude
)

/home/pchevali/.local/lib/python3.12/site-packages/xarray/core/dataarray.py:6318: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmin(dim, axis, keep_attrs, skipna)


In [16]:
def MSE_compute_and_plot(dataset,region=france_indexs,n=1,title=None):
    highest_member = get_highest_index(
        dataset["2t"].isel(values=paris_index), n=20
    )[n-1]
    
    tp = (dataset["tp"] * 1000).squeeze()  # millimeters

    
    MSE_500_star, MSE_s = moist_static_energy(
        dataset["2t"], dataset["t_500"], dataset["2d"], dataset["sp"], dataset["z_500"]
    )
    x_axis = dataset["tp"].step.values / np.timedelta64(1, "D")
    
    mse_500_plot = MSE_500_star.isel(values=region, number=highest_member).mean("values").squeeze().compute()/1000
    mse_s_plot = MSE_s.isel(values=region, number=highest_member).mean("values").squeeze().compute()/1000
    tp_plot = tp.isel(values=region, number=highest_member).mean("values").squeeze().compute()
    T_s_plot = dataset["2t"].isel(values=region, number=highest_member).mean("values").squeeze().compute() - 273.15
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [1.5, 1, 1]})
    
    #### MSE plot
    
    ax1.plot(x_axis, mse_s_plot, color="limegreen", label="$MSE_s$ (Surface)")
    ax1.plot(x_axis, mse_500_plot, color="darkgreen", label="$MSE_{500}^*$ (500hPa Saturation)")
    ax1.fill_between(x_axis, mse_s_plot, mse_500_plot, where=(mse_s_plot > mse_500_plot), 
                     color="red", interpolate=True, alpha=0.1, label="Instability") # periods where mse_s-mse_500>0
    
    ax1.set_ylabel("Moist Static Energy (J/g)", fontsize="large")
    ax1.grid(ls=":") 
    ax1.legend(loc="best", frameon=True, fontsize="large")
        
    # precipitation
    ax2.fill_between(x_axis, tp_plot, 0, color="blue", alpha=0.3)
    ax2.plot(x_axis, tp_plot, color="blue", label="Total Precipitation")
    ax2.set_ylabel("Precipitation (mm/6h)", fontsize="large")
    ax2.grid(ls=":")
    
    # temperature
    ax3.plot(x_axis, T_s_plot, color="black", label="Surface Temperature ($T_s$)")
    ax3.set_ylabel("Temperature (°C)", fontsize="large")
    ax3.grid(ls=":")
    ax3.set_xlabel("Time (Days)", fontsize="x-large")
    ax3.set_xlim(x_axis.min(), x_axis.max())

    fig.suptitle(title, fontsize="x-large")
    #plt.tight_layout()
    plt.savefig(f"atmospheric_profile_{title.replace(' ', '_')}.pdf",bbox_inches="tight")
    plt.show()

In [ ]:
paris_index

# Plots

### Trajectories

Compute the amount of trajectories that reach higher temps than ERA5

In [52]:
maxs_2003_paris=datasets_results['2003072300']['2t'].isel(values=paris_index).max("step").values
maxs_2019_paris=datasets_results['2019071300']['2t'].isel(values=paris_index).max("step").values
maxs_2025_paris=datasets_results['2025062000']['2t'].isel(values=paris_index).max("step").values
maxs_2003_paris_era5=era5_200307_08.sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")["t2m"].max().values
maxs_2019_paris_era5=era5_201907_08.sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")["t2m"].max().values
maxs_2025_paris_era5=era5_202506_07.sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")["t2m"].max().values

In [53]:
print("Au point de grille le plus proche de Paris")
print(f"Nombre de trajectoires dépassant le max d'ERA5 ({maxs_2003_paris_era5-273.15:.2f}°C) pour la période donnée (23/07/2003) : {np.sum(maxs_2003_paris>maxs_2003_paris_era5)}/{len(maxs_2003_paris)} ({np.mean(maxs_2003_paris>maxs_2003_paris_era5)*100:.2f}%) (max={np.max(maxs_2003_paris)-273.15:.2f}°C) | {np.sum(maxs_2003_paris>maxs_2019_paris_era5)}/{len(maxs_2003_paris)} ({np.mean(maxs_2003_paris>maxs_2019_paris_era5)*100:.2f}%) dépassant le record de 2019")
print(f"Nombre de trajectoires dépassant le max d'ERA5 ({maxs_2019_paris_era5-273.15:.2f}°C) pour la période donnée (13/07/2019) : {np.sum(maxs_2019_paris>maxs_2019_paris_era5)}/{len(maxs_2019_paris)} ({np.mean(maxs_2019_paris>maxs_2019_paris_era5)*100:.2f}%) (max={np.max(maxs_2019_paris)-273.15:.2f}°C) | {np.sum(maxs_2019_paris>maxs_2019_paris_era5)}/{len(maxs_2003_paris)} ({np.mean(maxs_2019_paris>maxs_2019_paris_era5)*100:.2f}%) dépassant le record de 2019")
print(f"Nombre de trajectoires dépassant le max d'ERA5 ({maxs_2025_paris_era5-273.15:.2f}°C) pour la période donnée (20/06/2025) : {np.sum(maxs_2025_paris>maxs_2025_paris_era5)}/{len(maxs_2025_paris)} ({np.mean(maxs_2025_paris>maxs_2025_paris_era5)*100:.2f}%) (max={np.max(maxs_2025_paris)-273.15:.2f}°C) | {np.sum(maxs_2025_paris>maxs_2019_paris_era5)}/{len(maxs_2003_paris)} ({np.mean(maxs_2025_paris>maxs_2019_paris_era5)*100:.2f}%) dépassant le record de 2019")

Au point de grille le plus proche de Paris
Nombre de trajectoires dépassant le max d'ERA5 (36.42°C) pour la période donnée (23/07/2003) : 3/129 (2.33%) (max=37.12°C) | 0/129 (0.00%) dépassant le record de 2019
Nombre de trajectoires dépassant le max d'ERA5 (40.78°C) pour la période donnée (13/07/2019) : 0/129 (0.00%) (max=38.75°C) | 0/129 (0.00%) dépassant le record de 2019
Nombre de trajectoires dépassant le max d'ERA5 (37.77°C) pour la période donnée (20/06/2025) : 10/129 (7.75%) (max=40.98°C) | 1/129 (0.78%) dépassant le record de 2019


In [54]:
maxs_2003_france=datasets_results['2003072300']['2t'].isel(values=france_indexs).mean("values").max("step").values
maxs_2019_france=datasets_results['2019071300']['2t'].isel(values=france_indexs).mean("values").max("step").values
maxs_2025_france=datasets_results['2025062000']['2t'].isel(values=france_indexs).mean("values").max("step").values
maxs_2003_france_era5=era5_200307_08.sel(latitude=france_latitude_bnd, longitude=france_longitude_bnd).mean(["latitude","longitude"])["t2m"].max().values
maxs_2019_france_era5=era5_201907_08.sel(latitude=france_latitude_bnd, longitude=france_longitude_bnd).mean(["latitude","longitude"])["t2m"].max().values
maxs_2025_france_era5=era5_202506_07.sel(latitude=france_latitude_bnd, longitude=france_longitude_bnd).mean(["latitude","longitude"])["t2m"].max().values

In [55]:
print("Moyenné sur la France")
print(f"Nombre de trajectoires dépassant le max d'ERA5 ({maxs_2003_france_era5-273.15:.2f}°C) pour la période donnée (23/07/2003) : {np.sum(maxs_2003_france>maxs_2003_france_era5)}/{len(maxs_2003_france)} ({np.mean(maxs_2003_france>maxs_2003_france_era5)*100:.2f}%) (max={np.max(maxs_2003_france)-273.15:.2f}°C) | {np.sum(maxs_2003_france>maxs_2019_france_era5)}/{len(maxs_2003_france)} ({np.mean(maxs_2003_france>maxs_2019_france_era5)*100:.2f}%) dépassant le record de 2019")
print(f"Nombre de trajectoires dépassant le max d'ERA5 ({maxs_2019_france_era5-273.15:.2f}°C) pour la période donnée (13/07/2019) : {np.sum(maxs_2019_france>maxs_2019_france_era5)}/{len(maxs_2019_france)} ({np.mean(maxs_2019_france>maxs_2019_france_era5)*100:.2f}%) (max={np.max(maxs_2019_france)-273.15:.2f}°C) | {np.sum(maxs_2019_france>maxs_2019_france_era5)}/{len(maxs_2003_france)} ({np.mean(maxs_2019_france>maxs_2019_france_era5)*100:.2f}%) dépassant le record de 2019")
print(f"Nombre de trajectoires dépassant le max d'ERA5 ({maxs_2025_france_era5-273.15:.2f}°C) pour la période donnée (20/06/2025) : {np.sum(maxs_2025_france>maxs_2025_france_era5)}/{len(maxs_2025_france)} ({np.mean(maxs_2025_france>maxs_2025_france_era5)*100:.2f}%) (max={np.max(maxs_2025_france)-273.15:.2f}°C) | {np.sum(maxs_2025_france>maxs_2019_france_era5)}/{len(maxs_2003_france)} ({np.mean(maxs_2025_france>maxs_2019_france_era5)*100:.2f}%) dépassant le record de 2019")

Moyenné sur la France
Nombre de trajectoires dépassant le max d'ERA5 (30.46°C) pour la période donnée (23/07/2003) : 1/129 (0.78%) (max=30.52°C) | 0/129 (0.00%) dépassant le record de 2019
Nombre de trajectoires dépassant le max d'ERA5 (31.00°C) pour la période donnée (13/07/2019) : 3/129 (2.33%) (max=32.64°C) | 3/129 (2.33%) dépassant le record de 2019
Nombre de trajectoires dépassant le max d'ERA5 (29.42°C) pour la période donnée (20/06/2025) : 28/129 (21.71%) (max=32.42°C) | 5/129 (3.88%) dépassant le record de 2019


Plots

#### Single point (Paris)

In [ ]:
check_chaos(
    (datasets_results['2003072300']['2t'].isel(values=paris_index) - 273.15),
    "",
    era=era5_200307_08.sel(
        latitude=paris_latitude, longitude=paris_longitude, method="nearest"
    )["t2m"]
    - 273.15,
    time_of_day=[12],
    save="example_boosting_2003_ts_paris"
)

In [ ]:
check_chaos(
    (datasets_results['2019071300'].isel(values=paris_index)["2t"] - 273.15),
    "",
    era=era5_201907_08.sel(
        latitude=paris_latitude, longitude=paris_longitude, method="nearest"
    )["t2m"]
    - 273.15,
    time_of_day=[12],
    save="example_boosting_2019_ts_paris"
)

In [ ]:
check_chaos(
    (datasets_results['2025062000']['2t'].isel(values=paris_index) - 273.15),
    "",
    era=era5_202506_07.sel(
        latitude=paris_latitude, longitude=paris_longitude, method="nearest"
    )["t2m"]
    - 273.15,
    time_of_day=[12],
    save="example_boosting_2025_ts_paris"
)

#### Over an Area (France)

In [ ]:
check_chaos(
    (datasets_results['2003072300']['2t'].isel(values=france_indexs).mean("values") - 273.15),
    "",
    era=era5_200307_08.sel(
        latitude=france_latitude_bnd, longitude=france_longitude_bnd)["t2m"].mean(["latitude","longitude"])
    - 273.15,
    time_of_day=[12],
    save="example_boosting_2003_ts_france_mean"
)

In [ ]:
check_chaos(
    (datasets_results['2019071300']['2t'].isel(values=france_indexs).mean("values") - 273.15),
    "",
    era=era5_201907_08.sel(
        latitude=france_latitude_bnd, longitude=france_longitude_bnd)["t2m"].mean(["latitude","longitude"])
    - 273.15,
    time_of_day=[12],
    save="example_boosting_2019_ts_france_mean"
)

In [ ]:
check_chaos(
    (datasets_results['2025062000']['2t'].isel(values=france_indexs).mean("values") - 273.15),
    "",
    era=era5_202506_07.sel(
        latitude=france_latitude_bnd, longitude=france_longitude_bnd)["t2m"].mean(["latitude","longitude"])
    - 273.15,
    time_of_day=[12],
    save="example_boosting_2025_ts_france_mean"
)

### Growth rates

In [ ]:
growth_rate_pairwise_bootstrap(
    data=datasets_results['2003072300']['z_500'],
    space_indexes=france_centered_indexs,
    times=[12],
    crps=True,
    save="example_boosting_2003_z500_60W60E6015S70N_growth_rate"
)

In [ ]:
growth_rate_pairwise_bootstrap(
    data=datasets_results['2019071300']['z_500'],
    space_indexes=france_centered_indexs,
    times=[12],
    crps=True,
    save="example_boosting_2019_z500_60W60E6015S70N_growth_rate"
)

In [ ]:
growth_rate_pairwise_bootstrap(
    data=datasets_results['2025062000']['z_500'],
    space_indexes=france_centered_indexs,
    times=[12],
    crps=True,
    save="example_boosting_2025_z500_60W60E6015S70N_growth_rate"
)

### Composite maps

In [ ]:
plot_extreme_events_composite(
    datasets_results=datasets_results,
    target=paris_index,
    region=plot_region_indexs,
    n=5,
    date_key="2025062000",
    var_main="2t",
    var_contour="z_500",
    save="test_boosting_2025_5_hottest",
)
plot_extreme_events_composite(
    datasets_results=datasets_results,
    target=paris_index,
    region=plot_region_indexs,
    n=5,
    date_key="2019071300",
    var_main="2t",
    var_contour="z_500",
    save="test_boosting_2019_5_hottest"
)
plot_extreme_events_composite(
    datasets_results=datasets_results,
    target=paris_index,
    region=plot_region_indexs,
    n=5,
    date_key="2003072300",
    var_main="2t",
    var_contour="z_500",
    save="test_boosting_2003_5_hottest"
)

### 15 hottest days era5 vs 15 hottest in simu

In [ ]:
mask_00=(era5_200307_08.time.dt.hour==0).values
mask_06=(era5_200307_08.time.dt.hour==6).values
mask_12=(era5_200307_08.time.dt.hour==12).values
mask_18=(era5_200307_08.time.dt.hour==18).values

In [ ]:
hottest_members_2003 = get_highest_index(
    datasets_results["2003072300"]["2t"].isel(values=paris_index), n=10
)[0]
paris_temp_selected_2003 = (
    datasets_results["2003072300"]["2t"]
    .isel(values=paris_index, number=hottest_members_2003)
    # .isel(step=mask_12)
    .squeeze()
)

hottest_members_2019 = get_highest_index(
    datasets_results["2019071300"]["2t"].isel(values=paris_index), n=1
)
paris_temp_selected_2019 = (
    datasets_results["2019071300"]["2t"]
    .isel(values=paris_index, number=hottest_members_2019)
#    .isel(step=mask_12)
    .squeeze()
)

hottest_members_2025 = get_highest_index(
    datasets_results["2025062000"]["2t"].isel(values=paris_index), n=1
)
paris_temp_selected_2025 = (
    datasets_results["2025062000"]["2t"]
    .isel(values=paris_index, number=hottest_members_2025)
    # .isel(step=mask_12)
    .squeeze()
)
paris_era5_2003 = (
    era5_200307_08["t2m"]
    .sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")
    # .isel(time=mask_12)
)

paris_era5_2025 = (
    era5_202506_07["t2m"]
    .sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")
    # .isel(time=mask_12)
)
paris_era5_2019 = (
    era5_201907_08["t2m"]
    .sel(latitude=paris_latitude, longitude=paris_longitude, method="nearest")
    # .isel(time=mask_12)
)
data = [
    paris_temp_selected_2003.values - 273.15,
    paris_era5_2003.values - 273.15,
    paris_temp_selected_2019.values - 273.15,
    paris_era5_2019.values - 273.15,
    paris_temp_selected_2025.values - 273.15,
    paris_era5_2025.values - 273.15,
]
data = [np.sort(prout)[-10:] for prout in data]
labels = [
    "2003 (Sim)", "2003 (ERA5)",
    "2019 (Sim)", "2019 (ERA5)",
    "2025 (Sim)", "2025 (ERA5)",
]
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["black", "blue"] * 3

for i, (y_vals, color) in enumerate(zip(data, colors)):
    x_vals = np.random.normal(loc=i, scale=0.05, size=len(y_vals)) # Adds jitter
    ax.scatter(x_vals, y_vals, color=color, alpha=0.8)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize="x-large")

ax.set_ylabel("Temperature (°C)", fontsize="x-large")
ax.set_title("Hottest Temperatures in Paris: AIFS-CRPS vs ERA5", fontsize="x-large")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle="--")
ax.set_axisbelow(True)

sim_patch = mpatches.Patch(color="black", label="AIFS-CRPS (Sim)")
era_patch = mpatches.Patch(color="blue", label="ERA5")
ax.legend(handles=[sim_patch, era_patch], frameon=True, loc="upper left", fontsize="x-large")

plt.tight_layout()
plt.savefig("boxplot_temperature_paris.pdf",bbox_inches="tight")
plt.show()

In [ ]:
hottest_members_2003 = get_highest_index(
    datasets_results["2003072300"]["2t"].isel(values=france_indexs).mean("values"), n=10
)[0]
paris_temp_selected_2003 = (
    datasets_results["2003072300"]["2t"]
    .isel(values=france_indexs, number=hottest_members_2003)
    .mean("values")
    # .isel(step=mask_12)
    .squeeze()
)

hottest_members_2019 = get_highest_index(
    datasets_results["2019071300"]["2t"].isel(values=paris_index), n=1
)
paris_temp_selected_2019 = (
    datasets_results["2019071300"]["2t"]
    .isel(values=france_indexs, number=hottest_members_2019)
    .mean("values")
#    .isel(step=mask_12)
    .squeeze()
)

hottest_members_2025 = get_highest_index(
    datasets_results["2025062000"]["2t"].isel(values=paris_index), n=1
)
paris_temp_selected_2025 = (
    datasets_results["2025062000"]["2t"]
    .isel(values=france_indexs, number=hottest_members_2025)
    .mean("values")
    # .isel(step=mask_12)
    .squeeze()
)
paris_era5_2003 = (
    era5_200307_08["t2m"]
    .sel(latitude=france_latitude_bnd, longitude=france_longitude_bnd)
    .mean(["latitude","longitude"])
    # .isel(time=mask_12)
)

paris_era5_2025 = (
    era5_202506_07["t2m"]
    .sel(latitude=france_latitude_bnd, longitude=france_longitude_bnd)
    .mean(["latitude","longitude"])
    # .isel(time=mask_12)
)
paris_era5_2019 = (
    era5_201907_08["t2m"]
    .sel(latitude=france_latitude_bnd, longitude=france_longitude_bnd)
    .mean(["latitude","longitude"])
    # .isel(time=mask_12)
)
data = [
    paris_temp_selected_2003.values - 273.15,
    paris_era5_2003.values - 273.15,
    paris_temp_selected_2019.values - 273.15,
    paris_era5_2019.values - 273.15,
    paris_temp_selected_2025.values - 273.15,
    paris_era5_2025.values - 273.15,
]
data = [np.sort(prout)[-10:] for prout in data]
labels = [
    "2003 (Sim)", "2003 (ERA5)",
    "2019 (Sim)", "2019 (ERA5)",
    "2025 (Sim)", "2025 (ERA5)",
]
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["black", "blue"] * 3

for i, (y_vals, color) in enumerate(zip(data, colors)):
    x_vals = np.random.normal(loc=i, scale=0.05, size=len(y_vals)) # Adds jitter
    ax.scatter(x_vals, y_vals, color=color, alpha=0.8)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize="x-large")

ax.set_ylabel("Temperature (°C)", fontsize="x-large")
ax.set_title("Hottest Temperatures averaged over France: AIFS-CRPS vs ERA5", fontsize="x-large")
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle="--")
ax.set_axisbelow(True)

sim_patch = mpatches.Patch(color="black", label="AIFS-CRPS (Sim)")
era_patch = mpatches.Patch(color="blue", label="ERA5")
ax.legend(handles=[sim_patch, era_patch], frameon=True, loc="upper left", fontsize="x-large")


plt.tight_layout()
plt.savefig("boxplot_temperature_france.pdf",bbox_inches="tight")
plt.show()

### MSE

In [ ]:
MSE_compute_and_plot(
    datasets_results["2003072300"], title="Hottest member of the 2003 simulation"
)
MSE_compute_and_plot(
    datasets_results["2003072300"],
    n=2,
    title="2nd hottest member of the 2003 simulation",
)
MSE_compute_and_plot(
    datasets_results["2003072300"],
    n=3,
    title="3rd hottest member of the 2003 simulation",
)

MSE_compute_and_plot(
    datasets_results["2019071300"], title="Hottest member of the 2019 simulation"
)
MSE_compute_and_plot(
    datasets_results["2019071300"],
    n=2,
    title="2nd hottest member of the 2019 simulation",
)
MSE_compute_and_plot(
    datasets_results["2019071300"],
    n=3,
    title="3rd hottest member of the 2019 simulation",
)

MSE_compute_and_plot(
    datasets_results["2025062000"], title="Hottest member of the 2025 simulation"
)
MSE_compute_and_plot(
    datasets_results["2025062000"],
    n=2,
    title="2nd hottest member of the 2025 simulation",
)
MSE_compute_and_plot(
    datasets_results["2025062000"],
    n=3,
    title="3rd hottest member of the 2025 simulation",
)

### Maps as gif